In [ ]:
import os
import numpy as np
import pandas as pd

# ===============================================================
# CONFIG
# ===============================================================
ATTACK_FILE = "./dataset/SWaT_Dataset_Attack_v0.csv"

OUT_DIR = "./preprocessed_swat_supervised/"
os.makedirs(OUT_DIR, exist_ok=True)

DROP_COLS = ["Timestamp", "Normal/Attack"]  # label géré à part

# ---- Winsorization (clip quantiles) on RAW values (before MinMax)
USE_WINSOR = True
W_Q_LOW  = 0.001   # 0.1%
W_Q_HIGH = 0.999   # 99.9%

# ===============================================================
# MinMax scaler "maison" (fit GLOBAL only)
# ===============================================================
def fit_minmax(X: np.ndarray):
    mn = np.min(X, axis=0)
    mx = np.max(X, axis=0)
    return mn.astype(np.float32), mx.astype(np.float32)

def transform_minmax(X: np.ndarray, mn: np.ndarray, mx: np.ndarray, eps: float = 1e-12):
    denom = (mx - mn)
    denom = np.where(denom < eps, 1.0, denom)
    return ((X - mn) / denom).astype(np.float32)

def pct_out01(X_scaled: np.ndarray) -> float:
    return float(((X_scaled < 0.0) | (X_scaled > 1.0)).mean()) * 100.0

# ===============================================================
# Winsorization helpers (RAW domain)
# ===============================================================
def compute_winsor_bounds_raw(X_train_raw: np.ndarray, q_low: float = 0.001, q_high: float = 0.999):
    """
    Compute per-feature quantile bounds on GLOBAL RAW (Attack CSV) only.
    """
    low = np.quantile(X_train_raw, q_low, axis=0)
    high = np.quantile(X_train_raw, q_high, axis=0)

    # safety: ensure low<=high
    low = np.minimum(low, high).astype(np.float32)
    high = np.maximum(low, high).astype(np.float32)
    return low, high

def apply_winsor_raw(X_raw: np.ndarray, low: np.ndarray, high: np.ndarray):
    """
    Clip per-feature in RAW domain.
    """
    return np.clip(X_raw, low, high).astype(np.float32)

# ===============================================================
# Loader SWaT
# ===============================================================
def load_csv_swat(csv_path: str, keep_label: bool):
    df = pd.read_csv(csv_path, sep=None, engine="python")

    y = None
    if keep_label and "Normal/Attack" in df.columns:
        y = (df["Normal/Attack"].astype(str).str.strip() == "Attack").astype(np.uint8).values

    # drop non-features
    df = df.drop(columns=DROP_COLS, errors="ignore")

    # virgules -> points, puis float
    for c in df.columns:
        df[c] = df[c].astype(str).str.replace(",", ".", regex=False)

    # to numeric + impute safety
    df = df.apply(pd.to_numeric, errors="coerce")
    df = df.ffill().bfill().fillna(0.0)
    df = df.replace([np.inf, -np.inf], 0.0)

    X = df.to_numpy(dtype=np.float32)
    return X, y, list(df.columns)

def main():
    # ---- load attack CSV as GLOBAL (normal + attack)
    Xg_raw, y_global, cols = load_csv_swat(ATTACK_FILE, keep_label=True)
    if y_global is None:
        raise RuntimeError("Colonne 'Normal/Attack' introuvable dans ATTACK_FILE.")

    print("GLOBAL raw:", Xg_raw.shape, "| y_global:", y_global.shape,
          "| NaN:", int(np.isnan(Xg_raw).sum()),
          "| Inf:", int(np.isinf(Xg_raw).sum()),
          "| min/max:", float(np.nanmin(Xg_raw)), float(np.nanmax(Xg_raw)))
    print("y_global counts:", {0: int((y_global == 0).sum()), 1: int((y_global == 1).sum())})
    print("F =", len(cols))

    # -----------------------------------------------------------
    # winsorization on RAW values (fit on GLOBAL only)
    # -----------------------------------------------------------
    if USE_WINSOR:
        w_low, w_high = compute_winsor_bounds_raw(Xg_raw, q_low=W_Q_LOW, q_high=W_Q_HIGH)
        Xg_raw_w = apply_winsor_raw(Xg_raw, w_low, w_high)

        print(f"\n✔ Winsor RAW enabled: q_low={W_Q_LOW} q_high={W_Q_HIGH}")
        print("RAW global min/max before:", float(Xg_raw.min()), float(Xg_raw.max()))
        print("RAW global min/max after :", float(Xg_raw_w.min()), float(Xg_raw_w.max()))

        Xg_raw = Xg_raw_w

        np.save(os.path.join(OUT_DIR, "winsor_low_raw.npy"),  w_low)
        np.save(os.path.join(OUT_DIR, "winsor_high_raw.npy"), w_high)

    # ---- fit scaler on GLOBAL only (raw possibly winsorized)
    mn, mx = fit_minmax(Xg_raw)

    # ---- transform (GLOBAL)
    X_global = transform_minmax(Xg_raw, mn, mx)

    # ---- sanity
    print("\nX_global:", X_global.shape, X_global.dtype,
          "| NaN:", int(np.isnan(X_global).sum()), "| Inf:", int(np.isinf(X_global).sum()),
          "| min/max:", float(X_global.min()), float(X_global.max()),
          "| out[0,1]%:", f"{pct_out01(X_global):.6f}%")

    if np.isnan(X_global).any():
        raise RuntimeError("Still NaNs after preprocessing.")

    # ---- save (order preserved as in CSV)
    np.save(os.path.join(OUT_DIR, "X_global.npy"), X_global)
    np.save(os.path.join(OUT_DIR, "y_global.npy"), y_global.astype(np.uint8))
    np.save(os.path.join(OUT_DIR, "scaler_min.npy"), mn)
    np.save(os.path.join(OUT_DIR, "scaler_max.npy"), mx)
    pd.Series(cols).to_csv(os.path.join(OUT_DIR, "feature_columns.csv"), index=False, header=False)

    print("\n✔ Saved to:", OUT_DIR)

if __name__ == "__main__":
    main()